In [ ]:
# ============ 模块0 依赖检查（原 Kaggle 模板 cell 已移除）============
# Kaggle 环境自带这些库；本地首次运行前请先执行：
#   pip install numpy pandas torch nibabel matplotlib scikit-learn
try:
    import numpy
    import pandas
    import torch
    import nibabel
    import matplotlib
    print("依赖检查通过：numpy / pandas / torch / nibabel / matplotlib 均已安装")
except ImportError as e:
    print("缺少依赖:", e)
    print("请在终端执行: pip install numpy pandas torch nibabel matplotlib scikit-learn")


In [ ]:
# %% ========================= 模块1 Environment =========================
import os
import glob
import random
from functools import lru_cache
import torch.nn as nn
 
import numpy as np
import nibabel as nib          # 若Kaggle环境未预装，取消下一行注释安装
# !pip install nibabel
 
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
 
 
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 
 
SEED = 42
set_seed(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
 
# ============ 环境自适应配置：同一份代码，Kaggle / 本地都能跑 ============
IS_KAGGLE = os.path.exists("/kaggle/input")
 
if IS_KAGGLE:
    DATA_ROOT = "/kaggle/input/datasets/user123454321/kits19-1"
    OUTPUT_DIR = "/kaggle/working"
    MAX_CASES  = None                        # Kaggle：全部病例
    NUM_WORKERS = 2
    EPOCHS     = 50
    BATCH_SIZE = 8
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
else:
    # ===== 本地快速验证模式（只验证逻辑，参数很小）=====
    DATA_ROOT = r"E:\大创\kits19_small"    # <-- 改成本地放小数据集的文件夹
    OUTPUT_DIR = r"E:\大创\output"         # 本地结果输出目录
    MAX_CASES  = 4                           # 本地快速验证：取前 4 个病例
    NUM_WORKERS = 0                          # Windows 本地必须为 0
    EPOCHS     = 1                           # 本地只跑 1 个 epoch
    BATCH_SIZE = 2
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.5, 0.25, 0.25   # 本地 4 病例：2 训练 / 1 验证 / 1 测试
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"运行环境: {'Kaggle' if IS_KAGGLE else '本地'}")
print(f"数据路径: {DATA_ROOT}")
print(f"输出目录: {OUTPUT_DIR}")


In [ ]:
# %% ========================= 模块2 Config =========================
class Config:
    # 数据路径（来自模块1的环境自适应配置）
    DATA_ROOT = DATA_ROOT
    OUTPUT_DIR = OUTPUT_DIR
 
    # 数据集划分比例（按病例划分，避免同一病例的切片同时出现在train/val/test）
    TRAIN_RATIO = TRAIN_RATIO
    VAL_RATIO = VAL_RATIO
    TEST_RATIO = TEST_RATIO
 
    # 本地快速验证时只取前几个病例
    MAX_CASES = MAX_CASES
    NUM_WORKERS = NUM_WORKERS
 
    # 图像与标签
    IMG_SIZE = 256
    NUM_CLASSES = 3          # 0=背景, 1=肾脏, 2=肿瘤
    HU_MIN = -200            # CT窗宽窗位裁剪下界
    HU_MAX = 300              # CT窗宽窗位裁剪上界
 
    # 训练超参数（本地小数据模式已在模块1中调小）
    BATCH_SIZE = BATCH_SIZE
    LEARNING_RATE = 1e-4
    EPOCHS = EPOCHS
 
    # 预留给后续贝叶斯/不确定性模块，基础U-Net阶段暂不使用
    DROPOUT_RATE = 0.0
    UNCERTAINTY_THRESHOLD = None
 
    SEED = SEED
    DEVICE = DEVICE
 
 
cfg = Config()


In [ ]:
# %% ========================= 模块3 Dataset =========================
def _find_volume(case_dir: str, name: str):
    """在病例目录中查找 imaging/segmentation 文件，兼容 .nii 和 .nii.gz"""
    for ext in (".nii", ".nii.gz"):
        vol = os.path.join(case_dir, name + ext)
        if os.path.isfile(vol):
            return vol
    return None


def scan_valid_cases(data_root: str, max_cases: int = None):
    """扫描data_root下所有case_*目录，保留同时具有imaging和segmentation的病例（兼容.nii/.nii.gz）"""
    case_dirs = sorted(glob.glob(os.path.join(data_root, "case_*")))
    valid_cases = [
        c for c in case_dirs
        if _find_volume(c, "imaging") is not None
        and _find_volume(c, "segmentation") is not None
    ]
    if max_cases is not None and len(valid_cases) > max_cases:
        valid_cases = valid_cases[:max_cases]
    print(f"数据集扫描完成: 共 {len(valid_cases)} 个有效病例（来自 {len(case_dirs)} 个目录）")
    return valid_cases
 
 
class KiTS19SliceDataset(Dataset):
    """
    从KiTS19病例的3D CT体积中提取2D轴向切片（假定数组形状为 [切片数, H, W]）。
    只保留含有肾脏(1)或肿瘤(2)标注的切片，过滤纯背景切片。
 
    初始化时一次性读盘、完成窗宽窗位归一化与resize，并把结果常驻内存；
    __getitem__ 只从内存取数据，不再重复读盘。
    （此前每次__getitem__现读整卷CT、缓存位只有4个，在shuffle训练下几乎每张切片
    都要重新读一次约1.2GB的体数据，是导致训练卡住不动的根本原因，这里做了修正）
    """
 
    def __init__(self, case_dirs, img_size, hu_min, hu_max, transform=None):
        self.img_size = img_size
        self.transform = transform
        self.images = []  # 每个元素: (img_size, img_size) float32
        self.masks = []   # 每个元素: (img_size, img_size) int64
        self._preload(case_dirs, hu_min, hu_max)
 
    def _preload(self, case_dirs, hu_min, hu_max):
        for case_dir in case_dirs:
            img_vol = nib.load(_find_volume(case_dir, "imaging")).get_fdata()
            seg_vol = nib.load(_find_volume(case_dir, "segmentation")).get_fdata()
            nonempty_idx = np.where(seg_vol.sum(axis=(1, 2)) > 0)[0]
 
            for idx in nonempty_idx:
                img = img_vol[idx, :, :].astype(np.float32)
                mask = seg_vol[idx, :, :].astype(np.int64)
 
                img = np.clip(img, hu_min, hu_max)
                img = (img - hu_min) / (hu_max - hu_min)
 
                img_t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
                mask_t = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).float()
 
                img_t = F.interpolate(img_t, size=(self.img_size, self.img_size),
                                       mode="bilinear", align_corners=False)
                mask_t = F.interpolate(mask_t, size=(self.img_size, self.img_size), mode="nearest")
 
                self.images.append(img_t.squeeze(0).squeeze(0).numpy().astype(np.float32))
                self.masks.append(mask_t.squeeze(0).squeeze(0).numpy().astype(np.int64))
 
        print(f"切片预加载完成: 共 {len(self.images)} 个有效切片（已过滤纯背景切片，已常驻内存）")
 
    def __len__(self):
        return len(self.images)
 
    def __getitem__(self, i):
        img_t = torch.from_numpy(self.images[i]).unsqueeze(0)  # (1, H, W)
        mask_t = torch.from_numpy(self.masks[i])                # (H, W)
 
        if self.transform is not None:
            img_t, mask_t = self.transform(img_t, mask_t)
 
        return img_t, mask_t


In [ ]:
# %% ========================= 模块4 DataLoader =========================
def split_cases(case_dirs, train_ratio, val_ratio, seed):
    """按病例（而非切片）划分train/val/test，避免同一病例的切片跨集合造成数据泄漏"""
    rng = random.Random(seed)
    cases = case_dirs.copy()
    rng.shuffle(cases)
    n = len(cases)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_cases = cases[:n_train]
    val_cases = cases[n_train:n_train + n_val]
    test_cases = cases[n_train + n_val:]
    return train_cases, val_cases, test_cases
 
 
all_cases = scan_valid_cases(cfg.DATA_ROOT, cfg.MAX_CASES)
train_cases, val_cases, test_cases = split_cases(
    all_cases, cfg.TRAIN_RATIO, cfg.VAL_RATIO, cfg.SEED
)
 
# 此处先不带增强构建数据集，训练集的增强由模块5单独挂载
train_dataset = KiTS19SliceDataset(train_cases, cfg.IMG_SIZE, cfg.HU_MIN, cfg.HU_MAX, transform=None)
val_dataset = KiTS19SliceDataset(val_cases, cfg.IMG_SIZE, cfg.HU_MIN, cfg.HU_MAX, transform=None)
test_dataset = KiTS19SliceDataset(test_cases, cfg.IMG_SIZE, cfg.HU_MIN, cfg.HU_MAX, transform=None)
 
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
                           num_workers=cfg.NUM_WORKERS, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)
 
print("=" * 50)
print("数据加载器统计")
print("=" * 50)
print(f"批次大小: {cfg.BATCH_SIZE}")
print(f"训练批次数: {len(train_loader)}")
print(f"验证批次数: {len(val_loader)}")
print(f"测试批次数: {len(test_loader)}")
print(f"总批次数: {len(train_loader) + len(val_loader) + len(test_loader)}")
print("=" * 50)
print(f"训练集病例数: {len(train_cases)} | 验证集病例数: {len(val_cases)} | 测试集病例数: {len(test_cases)}")


In [ ]:
# %% ========================= 模块5 Augmentation =========================
class TrainAugmentation:
    """
    训练阶段增强：随机水平翻转 / 随机垂直翻转 / 随机90度旋转，图像与mask同步变换。
    验证/测试阶段不做随机增强，只保留模块3中已完成的窗宽窗位归一化与resize。
    """
 
    def __call__(self, img, mask):
        if random.random() < 0.5:
            img = torch.flip(img, dims=[-1])
            mask = torch.flip(mask, dims=[-1])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[-2])
            mask = torch.flip(mask, dims=[-2])
        if random.random() < 0.5:
            k = random.choice([1, 2, 3])
            img = torch.rot90(img, k, dims=[-2, -1])
            mask = torch.rot90(mask, k, dims=[-2, -1])
        return img, mask
 
 
train_transform = TrainAugmentation()
train_dataset.transform = train_transform
 
# transform是Dataset的运行时属性，重建train_loader即可让新的transform生效
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
                           num_workers=cfg.NUM_WORKERS, drop_last=True)
 
print("数据增强已启用（仅训练集）: 随机水平翻转 / 随机垂直翻转 / 随机90°旋转")


In [ ]:

# %% ========================= 模块6 Model =========================
class DoubleConv(nn.Module):
    """(Conv3x3 -> BN -> ReLU) x2"""
 
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
 
    def forward(self, x):
        return self.block(x)
 
 
class UNet(nn.Module):
    """基础U-Net，输入单通道CT切片，输出num_classes通道的分割logits（不含贝叶斯/Dropout采样机制）"""
 
    def __init__(self, in_channels=1, num_classes=3, base_channels=32):
        super().__init__()
        c1, c2, c3, c4, c5 = base_channels, base_channels * 2, base_channels * 4, base_channels * 8, base_channels * 16
 
        self.enc1 = DoubleConv(in_channels, c1)
        self.enc2 = DoubleConv(c1, c2)
        self.enc3 = DoubleConv(c2, c3)
        self.enc4 = DoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)
 
        self.bottleneck = DoubleConv(c4, c5)
 
        self.up4 = nn.ConvTranspose2d(c5, c4, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c5, c4)
        self.up3 = nn.ConvTranspose2d(c4, c3, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c4, c3)
        self.up2 = nn.ConvTranspose2d(c3, c2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c3, c2)
        self.up1 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c2, c1)
 
        self.out_conv = nn.Conv2d(c1, num_classes, kernel_size=1)
 
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
 
        b = self.bottleneck(self.pool(e4))
 
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
 
        return self.out_conv(d1)  # (B, num_classes, H, W) logits
 
 
model = UNet(in_channels=1, num_classes=cfg.NUM_CLASSES, base_channels=32).to(cfg.DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"模型已构建: 基础U-Net | 参数量: {n_params:,}")

In [ ]:
 
# %% ========================= 模块7 Loss & Optimizer =========================
class SoftDiceLoss(nn.Module):
    """多类别可微Dice损失，对logits做softmax后与one-hot标签计算Dice，各类别取平均"""
 
    def __init__(self, num_classes, eps=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.eps = eps
 
    def forward(self, logits, target):
        probs = torch.softmax(logits, dim=1)
        target_onehot = F.one_hot(target, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
 
        dims = (0, 2, 3)
        intersection = torch.sum(probs * target_onehot, dims)
        union = torch.sum(probs + target_onehot, dims)
        dice_per_class = (2 * intersection + self.eps) / (union + self.eps)
        return 1.0 - dice_per_class.mean()
 
 
class CombinedLoss(nn.Module):
    """CrossEntropy + Dice 组合损失（等权重）。多类别分割用CE替代原方案中的BCE"""
 
    def __init__(self, num_classes):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = SoftDiceLoss(num_classes)
 
    def forward(self, logits, target):
        return self.ce(logits, target) + self.dice(logits, target)
 
 
criterion = CombinedLoss(cfg.NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
 
print(f"损失函数: CrossEntropy + Dice | 优化器: Adam(lr={cfg.LEARNING_RATE}) | 调度器: ReduceLROnPlateau")

In [ ]:

# %% ========================= 模块8 Training =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
 
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item() * imgs.size(0)
 
    return running_loss / len(loader.dataset)

In [ ]:
# %% ========================= 模块9 Validation + 主训练循环 =========================
def compute_dice_iou(pred, target, num_classes, eps=1e-6):
    """基于argmax硬预测计算各类别Dice和IoU，返回按类别平均值（含背景）"""
    dice_scores, iou_scores = [], []
    for c in range(num_classes):
        pred_c = (pred == c)
        target_c = (target == c)
        intersection = (pred_c & target_c).sum().item()
        pred_sum = pred_c.sum().item()
        target_sum = target_c.sum().item()
        union = pred_sum + target_sum - intersection
 
        if target_sum == 0 and pred_sum == 0:
            dice_scores.append(1.0)
            iou_scores.append(1.0)
        else:
            dice_scores.append((2 * intersection + eps) / (pred_sum + target_sum + eps))
            iou_scores.append((intersection + eps) / (union + eps))
 
    return float(np.mean(dice_scores)), float(np.mean(iou_scores))
 
 
@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes):
    model.eval()
    running_loss, dice_sum, iou_sum, n_batches = 0.0, 0.0, 0.0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss = criterion(logits, masks)
 
        preds = torch.argmax(logits, dim=1)
        dice, iou = compute_dice_iou(preds, masks, num_classes)
 
        running_loss += loss.item() * imgs.size(0)
        dice_sum += dice
        iou_sum += iou
        n_batches += 1
 
    return {
        "loss": running_loss / len(loader.dataset),
        "dice": dice_sum / n_batches,
        "iou": iou_sum / n_batches,
    }
 
 
# ---- 实际执行训练+验证循环（依赖模块8的train_one_epoch与本模块的evaluate）----
history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": []}
best_val_dice = -1.0
best_ckpt_path = os.path.join(cfg.OUTPUT_DIR, "best_unet.pth")
 
print("=" * 50)
print("开始基础U-Net模型训练（不含贝叶斯模块）")
print("=" * 50)
 
for epoch in range(1, cfg.EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, cfg.DEVICE)
    history["train_loss"].append(train_loss)
 
    if len(val_loader) > 0:
        val_metrics = evaluate(model, val_loader, criterion, cfg.DEVICE, cfg.NUM_CLASSES)
        scheduler.step(val_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_dice"].append(val_metrics["dice"])
        history["val_iou"].append(val_metrics["iou"])
 
        print(f"Epoch [{epoch}/{cfg.EPOCHS}] | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_metrics['loss']:.4f} | Val Dice: {val_metrics['dice']:.4f} | "
              f"Val IoU: {val_metrics['iou']:.4f}")
 
        if val_metrics["dice"] > best_val_dice:
            best_val_dice = val_metrics["dice"]
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"  -> 保存最优模型 (Val Dice: {best_val_dice:.4f})")
    else:
        # 验证集为空（本地快速验证 2 病例）：不写 NaN 占位值，避免画图/统计被污染
        # 同时保存一轮权重，保证后续推理模块有模型可加载
        print(f"Epoch [{epoch}/{cfg.EPOCHS}] | Train Loss: {train_loss:.4f} | (验证集为空，跳过验证)")
        if best_val_dice < 0.0:
            best_val_dice = 0.0
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"  -> 验证集为空，保存当前轮模型: {best_ckpt_path}")
 
print("=" * 50)
if history["val_dice"]:
    print(f"训练完成，最优验证Dice: {best_val_dice:.4f}，模型已保存至: {best_ckpt_path}")
else:
    print(f"训练完成（验证集为空，按最后轮保存），模型已保存至: {best_ckpt_path}")


In [ ]:
# %% ========================= 模块10 Inference =========================
if len(test_dataset) > 0:
    inference_model = UNet(in_channels=1, num_classes=cfg.NUM_CLASSES, base_channels=32).to(cfg.DEVICE)
    inference_model.load_state_dict(torch.load(best_ckpt_path, map_location=cfg.DEVICE))
    inference_model.eval()
    print(f"已加载最优模型权重: {best_ckpt_path}")
 
 
    @torch.no_grad()
    def predict_single(model, img, device):
        """img: (1, H, W) tensor -> pred: (H, W) numpy数组（类别索引）"""
        img = img.unsqueeze(0).to(device)
        logits = model(img)
        pred = torch.argmax(logits, dim=1).squeeze(0)
        return pred.cpu().numpy()
 
 
    def get_sample_predictions(model, dataset, device, n_samples=4, seed=SEED):
        """从数据集中随机抽样n_samples个切片，返回(原图, 原始mask, 预测mask)三元组列表"""
        rng = random.Random(seed)
        indices = rng.sample(range(len(dataset)), min(n_samples, len(dataset)))
        samples = []
        for idx in indices:
            img, mask = dataset[idx]
            pred = predict_single(model, img, device)
            samples.append((img.squeeze(0).numpy(), mask.numpy(), pred))
        return samples
 
 
    sample_predictions = get_sample_predictions(inference_model, test_dataset, cfg.DEVICE, n_samples=4)
    print(f"已从测试集随机抽取 {len(sample_predictions)} 个样本完成推理")
else:
    sample_predictions = []
    print("测试集为空（本地小数据模式），跳过推理模块")


In [ ]:
# %% ========================= 模块11 Visualization =========================
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
 
MASK_CMAP = ListedColormap(["black", "yellow", "red"])  # 0=背景, 1=肾脏, 2=肿瘤
 
 
def plot_training_curves(history, save_path=None):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    # 验证集可能为空（本地快速验证），此时只画训练曲线，避免空/NaN数据导致崩溃
    has_val = bool(history.get("val_loss")) and len(history["val_loss"]) == len(epochs)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # 左上: Loss曲线
    axes[0, 0].plot(epochs, history["train_loss"], "r-", label="Training Loss")
    if has_val:
        axes[0, 0].plot(epochs, history["val_loss"], "g-", label="Validation Loss")
        best_ep = int(np.argmin(history["val_loss"])) + 1
        axes[0, 0].scatter([best_ep], [history["val_loss"][best_ep - 1]], color="blue",
                            zorder=5, label=f"best epoch= {best_ep}")
    axes[0, 0].set_title("Training and Validation Loss")
    axes[0, 0].set_xlabel("Epochs")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()

    # 右上: Dice曲线
    if has_val:
        axes[0, 1].plot(epochs, history["val_dice"], "g-", label="Validation Dice")
        best_ep = int(np.argmax(history["val_dice"])) + 1
        axes[0, 1].scatter([best_ep], [history["val_dice"][best_ep - 1]], color="blue",
                            zorder=5, label=f"best epoch= {best_ep}")
        axes[0, 1].legend()
    axes[0, 1].set_title("Validation Dice Coefficient")
    axes[0, 1].set_xlabel("Epochs")
    axes[0, 1].set_ylabel("Dice")

    # 左下: IoU曲线
    if has_val:
        axes[1, 0].plot(epochs, history["val_iou"], "g-", label="Validation IoU")
        best_ep = int(np.argmax(history["val_iou"])) + 1
        axes[1, 0].scatter([best_ep], [history["val_iou"][best_ep - 1]], color="blue",
                            zorder=5, label=f"best epoch= {best_ep}")
        axes[1, 0].legend()
    axes[1, 0].set_title("Validation IoU Coefficient")
    axes[1, 0].set_xlabel("Epochs")
    axes[1, 0].set_ylabel("IoU")

    # 右下: 训练Loss单独放大查看收敛情况
    axes[1, 1].plot(epochs, history["train_loss"], "r-", label="Training Loss")
    best_ep = int(np.argmin(history["train_loss"])) + 1
    axes[1, 1].scatter([best_ep], [history["train_loss"][best_ep - 1]], color="blue",
                        zorder=5, label=f"best epoch= {best_ep}")
    axes[1, 1].set_title("Training Loss")
    axes[1, 1].set_xlabel("Epochs")
    axes[1, 1].set_ylabel("Loss")
    axes[1, 1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
def plot_prediction_samples(samples, save_path=None):
    """每行: Original Image | Original Mask | Prediction，与参考模板格式一致"""
    n = len(samples)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
 
    for i, (img, mask, pred) in enumerate(samples):
        axes[i, 0].imshow(img, cmap="gray")
        axes[i, 0].set_title("Original Image")
        axes[i, 0].axis("off")
 
        axes[i, 1].imshow(mask, cmap=MASK_CMAP, vmin=0, vmax=2)
        axes[i, 1].set_title("Original Mask")
        axes[i, 1].axis("off")
 
        axes[i, 2].imshow(pred, cmap=MASK_CMAP, vmin=0, vmax=2)
        axes[i, 2].set_title("Prediction")
        axes[i, 2].axis("off")
 
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
 
 
plot_training_curves(history, save_path=os.path.join(cfg.OUTPUT_DIR, "training_curves.png"))
if sample_predictions:
    plot_prediction_samples(sample_predictions, save_path=os.path.join(cfg.OUTPUT_DIR, "prediction_samples.png"))
 
if history["val_dice"]:
    print(f"最优验证Dice: {max(history['val_dice']):.4f} | 最优验证IoU: {max(history['val_iou']):.4f}")
else:
    print("验证集为空，跳过验证指标汇总")


In [ ]:
# ============ 模块12 评估：用 unified_metrics 批量评估测试集分割结果 ============
# 对接人员4 交付的统一指标库 src/metrics/unified_metrics.py；口径与交接说明见 docs/评估工具统筹与交接说明.md
import sys, os, csv
sys.path.insert(0, os.getcwd())          # 保证能 import src（notebook 默认 CWD=E:\大创）
from src.metrics.unified_metrics import calculate_all_metrics

if len(test_dataset) == 0:
    print("测试集为空（本地小数据模式），跳过评估模块12")
else:
    model_eval = inference_model              # 已加载 best_unet.pth（模块10）
    model_eval.eval()

    # 1) 在测试集上批量推理，得到 3 类预测（0=背景 1=肾脏 2=肿瘤）
    pred_all, target_all = [], []
    with torch.no_grad():
        for imgs, masks in test_loader:
            logits = model_eval(imgs.to(cfg.DEVICE))
            pred_all.append(torch.argmax(logits, dim=1).cpu().numpy())
            target_all.append(masks.numpy())
    pred_all = np.concatenate(pred_all)       # (N,H,W) 0/1/2
    target_all = np.concatenate(target_all)

    # 2) 统一指标库是"二值掩膜"口径：把肿瘤类(class 2)单独二值化（评估肾脏则把 2 改成 1）
    pred_tumor = (pred_all == 2).astype(np.float32)
    target_tumor = (target_all == 2).astype(np.float32)

    # 3) 熵图占位：人员3 的 mc_predict（贝叶斯）未交付前先传全 0，
    #    Dice/IoU/像素准确率是真实结果；熵/拒绝/覆盖/选择性准确率此时是占位值，待 mc_predict 落地后替换
    entropy_map = np.zeros_like(pred_tumor)
    tumor_mask = target_tumor

    res = calculate_all_metrics(pred_tumor, target_tumor, entropy_map, tumor_mask, reject_threshold=0.5)
    print("==== 测试集肿瘤分割评估（统一指标库）====")
    for k, v in res.items():
        print(f"{k:22s}: {v}")

    # 4) 导出标准表格（与 test_metrics.py 同款格式）
    csv_path = os.path.join(cfg.OUTPUT_DIR, "evaluation_metrics.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(res.keys()))
        w.writeheader()
        w.writerow(res)
    print("已保存:", csv_path)
